# Mynt Advanced Analytics Project for ML Engineers - Chor Eduarte


#### Problem Definition

Rossmann is Germany's second-largest drug store chain. You are provided with historical sales data for 1,115 Rossmann stores. The task is to forecast the "Sales" column. Note that some stores in the dataset were temporarily closed for refurbishment

#### Import Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt

#### Exploring the data

In [ ]:
# Load the datasets (ensure the date is parsed correctly)
train = pd.read_csv("train.csv", parse_dates=["Date"])
store = pd.read_csv("store.csv")


In [ ]:
# Display first few rows of each dataset
print(train.head())
print(store.head())

In [ ]:
# Merge on the Store column
data = pd.merge(train, store, on="Store", how="left")
print(data.info())
print(data.describe())

##### Sales Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data['Sales'], bins=50, kde=True)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

##### Time Series Trend

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(data.groupby("Date")["Sales"].sum())
plt.title("Total Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

In [ ]:
data['DayOfWeek'] = data['Date'].dt.dayofweek
plt.figure(figsize=(10, 6))
sns.boxplot(x="DayOfWeek", y="Sales", data=data)
plt.title("Sales Distribution by Day of Week")
plt.xlabel("Day of Week (0=Monday)")
plt.ylabel("Sales")
plt.show()

##### Data Cleaning

In [ ]:
data.dtypes

In [ ]:
print(data.describe(include='all'))
print(data.isnull().sum())

In [ ]:
# Convert StateHoliday to string for consistency
data['StateHoliday'] = data['StateHoliday'].astype(str)

In [ ]:
# Fill missing CompetitionDistance with median
data["CompetitionDistance"].fillna(data["CompetitionDistance"].median(), inplace=True)

# Fill missing CompetitionOpenSinceMonth and CompetitionOpenSinceYear.
data["CompetitionOpenSinceMonth"].fillna(0, inplace=True)
data["CompetitionOpenSinceYear"].fillna(0, inplace=True)


In [ ]:
# Fill missing values for Promo2 related fields
data["Promo2"].fillna(0, inplace=True)
data["Promo2SinceWeek"].fillna(0, inplace=True)
data["Promo2SinceYear"].fillna(0, inplace=True)
data["PromoInterval"].fillna("None", inplace=True)

In [ ]:
# For any other numeric fields have missing values, fill with median
num_cols = data.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    if data[col].isnull().sum() > 0:
        data[col].fillna(data[col].median(), inplace=True)

In [ ]:
data['StateHoliday'].unique()

In [ ]:
#perform encoding on StateHoliday column

data.loc[data['StateHoliday'] == '0', 'StateHoliday'] = 0
data.loc[data['StateHoliday'] == 'a', 'StateHoliday'] = 1
data.loc[data['StateHoliday'] == 'b', 'StateHoliday'] = 2
data.loc[data['StateHoliday'] == 'c', 'StateHoliday'] = 3

In [ ]:
data['StateHoliday'] = data['StateHoliday'].astype(int)

In [ ]:
data.dtypes

#### Feature Engineering

In [ ]:
#Fix Dates

data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day
data['WeekOfYear'] = data['Date'].dt.isocalendar().week.astype(int)
data['DayOfWeek'] = data['Date'].dt.dayofweek
data['IsWeekend'] = data['DayOfWeek'].isin([5,6]).astype(int)

In [ ]:
# one-hot encoding for StoreType and Assortment
categorical_features = ['StoreType', 'Assortment']
data = pd.get_dummies(data, columns=categorical_features, drop_first=True)

In [ ]:
#Create a binary indicator for competition activity
data['HasCompetition'] = data['CompetitionDistance'].notnull().astype(int)

In [ ]:
# Process promotion interval
# 1. If there are missing/null values, replace them with an empty string
data['PromoInterval'].fillna('', inplace=True)

# 2. Define the possible months
possible_months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# 3. Create a new column for each month
for month in possible_months:
    col_name = f"Promo_{month}"
    data[col_name] = data['PromoInterval'].apply(lambda x: 1 if month in x.split(',') else 0)

In [ ]:
# Drop columns
drop_cols = ['Date', 'PromoInterval']
data.drop(columns=drop_cols, inplace=True)



In [ ]:
# Drop duplicates if any
data.drop_duplicates(inplace=True)

In [ ]:
# Inspect the cleaned data
print(data.info())
print(data.head())

In [ ]:
data.to_csv("cleaned_data.csv", index=False)

### Machine Learning training

#### Splitting data

In [ ]:
# from sklearn.model_selection import train_test_split

# # Assuming Sales is the target
# X = data.drop("Sales", axis=1)
# y = data["Sales"]

# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#### Linear Regression

In [ ]:
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, mean_absolute_error

# # Train the model
# lr = LinearRegression()
# lr.fit(X_train, y_train)

# # Predict and evaluate
# y_pred_lr = lr.predict(X_val)
# rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
# mae_lr = mean_absolute_error(y_val, y_pred_lr)
# print("Linear Regression - RMSE: {:.2f}, MAE: {:.2f}".format(rmse_lr, mae_lr))

#### Random Forest

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# rf = RandomForestRegressor(n_estimators=100, random_state=42)
# rf.fit(X_train, y_train)

# y_pred_rf = rf.predict(X_val)
# rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
# mae_rf = mean_absolute_error(y_val, y_pred_rf)
# print("Random Forest - RMSE: {:.2f}, MAE: {:.2f}".format(rmse_rf, mae_rf))

#### XGBoost

In [ ]:
# import xgboost as xgb

# xgb_model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, n_estimators=100)
# xgb_model.fit(X_train, y_train)

# y_pred_xgb = xgb_model.predict(X_val)
# rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
# mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
# print("XGBoost - RMSE: {:.2f}, MAE: {:.2f}".format(rmse_xgb, mae_xgb))
